In [9]:
import os

In [10]:
import duckdb
import pandas as pd

# Connect to existing database
#con = duckdb.connect(r"C:\Users\shaog\OneDrive\Documents\DSE3101\mind-the-data-gap\data.duckdb")
con = duckdb.connect(r"C:\Users\wxcal\Desktop\Y3S2\DSE3101\mind-the-data-gap\data.duckdb")
DATA_DIR = r"C:\Users\wxcal\Desktop\Y3S2\DSE3101\data"
# con = duckdb.connect("data.duckdb")

In [11]:
# Check tables
print(con.execute("SHOW TABLES").fetchdf())

                     name
0             abt_pt_jrny
1             abt_pt_ride
2                bus_stop
3   jrny_st_cd_ride_st_cd
4             pass_typ_cd
5      patron_catg_id_num
6               pay_cd_21
7                 pt_jrny
8                 pt_ride
9        svc_grade_id_num
10             tkt_typ_cd
11         trnspt_mode_cd
12             vls_marker


## Cleaning pt_ride
1. Drop incomplete data
2. Pick a representative weekday (13/2/2025)
3. Drop irrelevant columns
4. Select relevant patron categories

In [12]:
# Check columns of pt_ride first
print(con.execute("DESCRIBE pt_ride").fetchdf())

           column_name column_type null   key default extra
0               BIZ_DT        DATE  YES  None    None  None
1          BUS_SVC_NUM      BIGINT  YES  None    None  None
2              CRD_NUM     VARCHAR  YES  None    None  None
3      DEST_LOC_ID_NUM      BIGINT  YES  None    None  None
4             ENTRY_DT        DATE  YES  None    None  None
5             ENTRY_TM        TIME  YES  None    None  None
6              EXIT_DT        DATE  YES  None    None  None
7              EXIT_TM        TIME  YES  None    None  None
8          JRNY_ID_NUM      BIGINT  YES  None    None  None
9      ORIG_LOC_ID_NUM      BIGINT  YES  None    None  None
10  PATRON_CATG_ID_NUM      BIGINT  YES  None    None  None
11              PAY_CD      BIGINT  YES  None    None  None
12       RIDE_DISC_AMT      DOUBLE  YES  None    None  None
13    RIDE_DIST_KM_CNT      DOUBLE  YES  None    None  None
14       RIDE_FARE_AMT      DOUBLE  YES  None    None  None
15         RIDE_ID_NUM      BIGINT  YES 

In [13]:
# Select representative weekday, drop incomplete rows and keep perfect rides
pt_ride_df = con.execute("""
    SELECT 
        CRD_NUM,
        JRNY_ID_NUM,
        BUS_SVC_NUM,
        ENTRY_DT,
        ENTRY_TM,
        EXIT_DT,
        EXIT_TM,
        ORIG_LOC_ID_NUM,
        DEST_LOC_ID_NUM,
        TRNSPT_MODE_CD,
        PATRON_CATG_ID_NUM,
        RIDE_FARE_AMT,
        RIDE_DIST_KM_CNT,
        RIDE_MIN_CNT,
        RIDE_ST_CD
    FROM pt_ride
    WHERE BIZ_DT = '2025-02-13'
      AND CRD_NUM IS NOT NULL
      AND ORIG_LOC_ID_NUM IS NOT NULL
      AND DEST_LOC_ID_NUM IS NOT NULL
      AND ENTRY_TM IS NOT NULL
      AND EXIT_TM IS NOT NULL
      AND RIDE_FARE_AMT IS NOT NULL
      AND RIDE_MIN_CNT IS NOT NULL
      AND RIDE_ST_CD = 1
""").df()
print(pt_ride_df.shape)
# this has dimensions (3727069, 15)

(3727069, 15)


In [14]:
# PLEASE ONLY RUN THIS ONCE TO PREVENT DUPLICATE COLS

# Checking patron category in pt_ride
# print(pt_ride_df['PATRON_CATG_ID_NUM'].value_counts().sort_index())

# Map to patron categories in mapping
patron_map = con.execute("SELECT * FROM patron_catg_id_num").df()

# Left join the patron categories into pt_ride_df
pt_ride_df = pt_ride_df.merge(patron_map, on='PATRON_CATG_ID_NUM', how='left')

# Left join pt_ride_df with transport mode
transport_map = con.execute("SELECT * FROM trnspt_mode_cd").df()
# print(transport_map)
pt_ride_df = pt_ride_df.merge(transport_map, on='TRNSPT_MODE_CD', how='left')

In [15]:
pt_ride_df.head()

,CRD_NUM,JRNY_ID_NUM,BUS_SVC_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,DEST_LOC_ID_NUM,TRNSPT_MODE_CD,PATRON_CATG_ID_NUM,RIDE_FARE_AMT,RIDE_DIST_KM_CNT,RIDE_MIN_CNT,RIDE_ST_CD,PATRON_CATG_DESC_TXT,TRNSPT_MODE_DESC_TXT
0,2200005487496,110732640883,<NA>,2025-02-13,18:38:23,2025-02-13,19:04:14,215,3,2,1,1.66,8.3,25.850,1,Adult,RTS
1,0240002779850,110732640058,88,2025-02-13,08:33:24,2025-02-13,08:40:54,3482,3467,1,3,0.00,1.7,7.500,1,Student,BUS
2,1700192323745,110732641494,88,2025-02-13,12:36:04,2025-02-13,13:27:24,3562,4646,1,1,1.89,13.2,51.333,1,Adult,BUS
3,2200005487496,110732640883,410,2025-02-13,19:26:12,2025-02-13,19:31:41,3438,3492,1,1,0.07,1.0,5.483,1,Adult,BUS
4,2230002033461,110732664904,<NA>,2025-02-13,22:57:21,2025-02-13,23:24:02,418,407,2,1,1.89,13.5,26.683,1,Adult,RTS


In [ ]:
# In current notebook — save
import pickle

with open(os.path.join(DATA_DIR, "pt_ride_df.pkl"), "wb") as f:
    pickle.dump(pt_ride_df, f)

In [17]:
con.close()

In [18]:
# # Don't Run this First
# # Select representative weekday, drop incomplete rows, keep perfect rides, filter for certain patron categories and join the mappings
# # filter out patron categories

# filtered_pt_ride = con.execute("""
#     SELECT 
#         pr.crd_num,
#         pr.jrny_id_num,
#         pr.bus_svc_num,
#         pr.entry_dt,
#         pr.entry_tm,
#         pr.exit_dt,
#         pr.exit_tm,
#         pr.orig_loc_id_num,
#         pr.dest_loc_id_num,
#         pr.trnspt_mode_cd,
#         m.trnspt_mode_desc_txt AS trnspt_mode,
#         pr.patron_catg_id_num,
#         p.patron_catg_desc_txt AS patron_catg,
#         pr.ride_fare_amt,
#         pr.ride_dist_km_cnt,
#         pr.ride_min_cnt,
#         pr.ride_st_cd,
#         st.st_cd_desc_txt AS ride_st
#     FROM pt_ride pr
# 	JOIN trnspt_mode_cd m using (trnspt_mode_cd)
# 	JOIN patron_catg_id_num p using (patron_catg_id_num)
# 	JOIN jrny_st_cd_ride_st_cd st on pr.ride_st_cd = st.st_cd
#     WHERE pr.biz_dt = '2025-02-13'
#       AND pr.crd_num IS NOT NULL
#       AND pr.orig_loc_id_num IS NOT NULL
#       AND pr.dest_loc_id_num IS NOT NULL
#       AND pr.entry_tm IS NOT NULL
#       AND pr.exit_tm IS NOT NULL
#       AND pr.ride_fare_amt IS NOT NULL
#       AND pr.ride_min_cnt IS NOT NULL
#       AND pr.ride_st_cd = 1
#       AND pr.patron_catg_id_num IN (1, 3, 4)
# """)

# #pd.read_sql("SELECT * FROM temp_results", con) # to directly manipulate df but I keep the sql form updated for now first

In [19]:
# # Dont run this
# cleaned_pt_ride_df = cleaned_pt_ride.fetchdf()
# print(cleaned_pt_ride_df.shape) # (3554596, 18)
# # print(cleaned_pt_ride.head())